# pow benchmark — results explorer

Loads every `results/bench-*.json` written by `repro.sh` / `pow-bench --json`,
builds a per-N summary table, plots elapsed-vs-N, and overlays the theoretical
exponential distribution onto the empirical CDF for a chosen N.

The notebook is the _only_ place opinion is mixed with the raw numbers — the
JSON dumps are the source of truth.


In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# notebook lives in analysis/, results in repo_root/results/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
RESULTS_DIR = REPO_ROOT / "results"
print(f"looking for results in: {RESULTS_DIR}")

In [ ]:
@dataclass
class Run:
    source: str  # filename
    host: str  # hostname extracted from filename
    backend: str  # ShaNi | Mb
    target_secs: float
    runs_per_n: int
    threads: int
    calibration: dict
    df: pd.DataFrame  # one row per (N, run_index)


def load_one(path: Path) -> Run:
    data = json.loads(path.read_text())
    # filename pattern: bench-<host>-<stamp>.json
    stem = path.stem
    host = stem.split("-", 2)[1] if stem.startswith("bench-") else "unknown"
    rows = []
    for r in data["results"]:
        for i, (att, el) in enumerate(zip(r["attempts"], r["elapsed_secs"], strict=True)):
            rows.append(
                {
                    "n_zeros": r["n_zeros"],
                    "run": i,
                    "attempts": att,
                    "elapsed": el,
                    "hps": att / el if el > 0 else 0.0,
                }
            )
    return Run(
        source=path.name,
        host=host,
        backend=data["config"]["backend"],
        target_secs=float(data["config"]["target_secs"]),
        runs_per_n=int(data["config"]["runs_per_n"]),
        threads=int(data["config"]["threads"]),
        calibration=data["calibration"],
        df=pd.DataFrame(rows),
    )


def discover() -> list[Run]:
    if not RESULTS_DIR.exists():
        return []
    return [load_one(p) for p in sorted(RESULTS_DIR.glob("bench-*.json"))]


runs = discover()
if not runs:
    print("No results yet. Run ./repro.sh first.")
else:
    for r in runs:
        n_min = int(r.df["n_zeros"].min())
        n_max = int(r.df["n_zeros"].max())
        print(
            f"{r.source}: host={r.host} backend={r.backend} threads={r.threads} "
            f"N={n_min}..{n_max} runs/N={r.runs_per_n}"
        )

## Per-N summary table

For each `(host, backend, N)`, show median, p50/p95, stddev of elapsed,
effective H/s, and how many runs landed in the bucket.


In [ ]:
def summarise(runs: list[Run]) -> pd.DataFrame:
    frames = []
    for r in runs:
        g = r.df.groupby("n_zeros")
        summary = pd.DataFrame(
            {
                "runs": g.size(),
                "median_s": g["elapsed"].median(),
                "mean_s": g["elapsed"].mean(),
                "p95_s": g["elapsed"].quantile(0.95),
                "max_s": g["elapsed"].max(),
                "stddev_s": g["elapsed"].std(),
                "eff_hps": g.apply(
                    lambda x: (
                        x["attempts"].sum() / x["elapsed"].sum() if x["elapsed"].sum() > 0 else 0.0
                    )
                ),
            }
        )
        summary["host"] = r.host
        summary["backend"] = r.backend
        frames.append(summary.reset_index())
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    return out[
        [
            "host",
            "backend",
            "n_zeros",
            "runs",
            "median_s",
            "mean_s",
            "p95_s",
            "max_s",
            "stddev_s",
            "eff_hps",
        ]
    ]


summary = summarise(runs)
summary

## Headline metric: max N fitting under target

Per (host, backend): highest N at which the **median** elapsed time stays
below `target_secs`. Median, not mean — see methodology note in README.


In [ ]:
def headline(runs: list[Run]) -> pd.DataFrame:
    rows = []
    for r in runs:
        median_per_n = r.df.groupby("n_zeros")["elapsed"].median()
        passing = median_per_n[median_per_n <= r.target_secs]
        max_n = int(passing.index.max()) if not passing.empty else None
        rows.append(
            {
                "host": r.host,
                "backend": r.backend,
                "target_s": r.target_secs,
                "max_n": max_n,
            }
        )
    return pd.DataFrame(rows)


headline(runs)

## Elapsed vs N (log-y, error bars = ±1σ)

Theoretical line: expected time per solve is `16^N / effective_hashrate`.
Big mismatches mean the solver isn't running at its calibrated hashrate
(usually due to coordination overhead at low N).


In [ ]:
if runs:
    _fig, ax = plt.subplots(figsize=(9, 5))
    for r in runs:
        g = r.df.groupby("n_zeros")["elapsed"]
        med = g.median()
        std = g.std().fillna(0.0)
        label = f"{r.host} / {r.backend}"
        ax.errorbar(med.index, med.values, yerr=std.values, marker="o", capsize=3, label=label)
    # target line
    ax.axhline(
        runs[0].target_secs,
        color="grey",
        linestyle="--",
        label=f"target {runs[0].target_secs:.0f}s",
    )
    ax.set_yscale("log")
    ax.set_xlabel("N (leading hex zeros)")
    ax.set_ylabel("elapsed, seconds (log)")
    ax.set_title("Solve time vs difficulty")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("no data — run ./repro.sh")

## Empirical vs theoretical distribution at a fixed N

Time-to-solve is geometric in attempts and approximately exponential in
wall-clock, with rate λ = effective_hps / 16^N. If our solver behaves
ideally, the empirical CDF should hug the theoretical one. Systematic
deviation points to coordination cost or hashrate inconsistency.


In [ ]:
def plot_cdf(run: Run, n: int) -> None:
    elapsed = run.df[run.df["n_zeros"] == n]["elapsed"].sort_values().to_numpy()
    if len(elapsed) < 2:
        print(f"not enough data at N={n} (have {len(elapsed)})")
        return
    eff_hps = run.df[run.df["n_zeros"] == n]["hps"].mean()
    rate = eff_hps / (16**n)
    ts = np.linspace(0, elapsed.max() * 1.2, 200)
    theoretical = 1.0 - np.exp(-rate * ts)

    _fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.step(
        elapsed,
        np.arange(1, len(elapsed) + 1) / len(elapsed),
        where="post",
        label=f"empirical (n={len(elapsed)})",
    )
    ax.plot(ts, theoretical, linestyle="--", label=f"theoretical Exp(rate≈{rate:.3f}/s)")
    ax.set_xlabel("elapsed, s")
    ax.set_ylabel("CDF")
    ax.set_title(f"{run.host} / {run.backend} — solve-time distribution at N={n}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


if runs:
    # pick the "most statistically meaningful" N — one with runs >= 10
    r = runs[0]
    counts = r.df.groupby("n_zeros").size()
    candidates = counts[counts >= 10]
    if not candidates.empty:
        plot_cdf(r, int(candidates.index.max()))
    else:
        plot_cdf(r, int(counts.index.max()))

## Backend speedup matrix

Pivot of median-elapsed by N and (host, backend). Ratios make it easy to
spot where multi-buffer wins or loses, and by how much.


In [ ]:
if runs:
    pivot = (
        summarise(runs)
        .assign(label=lambda d: d["host"] + "/" + d["backend"])
        .pivot_table(index="n_zeros", columns="label", values="median_s")
    )
    display(pivot)
    if pivot.shape[1] >= 2:
        baseline = pivot.iloc[:, 0]
        ratios = pivot.div(baseline, axis=0)
        print(f"speedup vs {pivot.columns[0]} (lower = faster):")
        display(ratios)